# 04.5 Reference Semantics and Aliasing

**Aliasing** is when two or more names refer to the same object. It is not a bug
— it is how Python works, and it is often exactly what you want. It becomes a bug
only when it is unintentional.

This notebook is about recognising aliasing before it surprises you.

## Theory

### Where aliases come from

Every one of these creates an alias rather than a copy:

```python
b = a                    # assignment
container.append(a)      # storing in a collection
function(a)              # passing as an argument
for item in items:       # the loop variable
d["key"] = a             # storing as a dict value
a, b = a, a              # unpacking
```

None of them copy. All of them add a reference to the same object.

### When aliasing is harmless

For **immutable** objects, aliasing can never cause a surprise. Nobody can change
the object, so sharing it is safe. This is why you never worry about aliasing an
`int` or a `str`.

For **mutable** objects, every alias is a route through which the object can
change.

### The rule that predicts everything

> A change is visible through every alias **if and only if** you mutated the
> object rather than rebound the name.

You met this in 04.2 for functions. It is the same rule everywhere.

### Detecting aliasing

`is` answers it directly. When debugging "why did this change?", compare `id()`
of the things you suspect.

In [ ]:
# Every one of these creates an alias, not a copy.
source = ["original"]

by_assignment = source
in_a_list = [source]
in_a_dict = {"key": source}
in_a_tuple = (source,)

print("Are all of these the same object?")
print("   assignment:      ", by_assignment is source)
print("   inside a list:   ", in_a_list[0] is source)
print("   inside a dict:   ", in_a_dict["key"] is source)
print("   inside a tuple:  ", in_a_tuple[0] is source)

# One mutation is visible through all of them.
source.append("added")

print("")
print("After source.append('added'):")
print("   by_assignment:", by_assignment)
print("   in_a_list:    ", in_a_list)
print("   in_a_dict:    ", in_a_dict)
print("   in_a_tuple:   ", in_a_tuple)

print("")
print("One object. Five routes to it.")

## Aliasing is invisible until something changes

The danger is that aliased code behaves identically to copied code — right up
until the first mutation.

In [ ]:
# These two look interchangeable.
aliased = [1, 2, 3]
alias = aliased

copied = [1, 2, 3]
copy_of_it = list(copied)

print("Before any change, both pairs look identical:")
print("   aliased:", aliased, " alias:", alias, " equal?", aliased == alias)
print("   copied: ", copied, " copy: ", copy_of_it, " equal?", copied == copy_of_it)

# Reading behaves the same.
print("")
print("Reading is identical:", alias[0], copy_of_it[0])

# Only mutation reveals the difference.
alias.append(99)
copy_of_it.append(99)

print("")
print("After appending 99 to the second name of each pair:")
print("   aliased:", aliased, "<- changed")
print("   copied: ", copied, "<- untouched")

print("")
print("The difference was there from the start. It was just not observable.")

## Aliasing in loops

The loop variable is an alias for each item in turn. Mutating through it changes
the collection; rebinding it does not.

In [ ]:
# Rebinding the loop variable changes nothing in the list.
rows = [[1], [2], [3]]

for row in rows:
    # This repoints the local name `row`. The list is untouched.
    row = ["replaced"]

print("After rebinding the loop variable:")
print("   rows:", rows, "<- unchanged")

# Mutating through the loop variable DOES change the list.
rows = [[1], [2], [3]]

for row in rows:
    # This mutates the object `row` refers to.
    row.append("added")

print("")
print("After mutating through the loop variable:")
print("   rows:", rows, "<- changed")

print("")
print("Same loop shape, opposite result. The difference is mutate vs rebind.")

# To replace items, assign by index.
rows = [[1], [2], [3]]

for index in range(len(rows)):
    rows[index] = ["replaced"]

print("")
print("To actually replace items, assign through the index:")
print("   rows:", rows)

## Aliasing across function boundaries

Combining 04.2 with aliasing: a function that stores its argument creates an
alias that outlives the call.

In [ ]:
class EventLog:
    """Stores whatever list it is handed - creating an alias."""

    def __init__(self, entries):
        # This does NOT copy. self.entries aliases the caller's list.
        self.entries = entries

    def add(self, message):
        self.entries.append(message)


class SafeEventLog:
    """Copies the incoming list, so the caller cannot interfere."""

    def __init__(self, entries):
        # list() builds a new list with the same items.
        self.entries = list(entries)

    def add(self, message):
        self.entries.append(message)


# The aliasing version shares state with the caller.
caller_list = ["first"]
unsafe = EventLog(caller_list)

caller_list.append("added by the caller")
unsafe.add("added by the log")

print("ALIASING version:")
print("   caller_list:  ", caller_list)
print("   log.entries:  ", unsafe.entries)
print("   same object?  ", unsafe.entries is caller_list)

# The copying version is independent.
caller_list = ["first"]
safe = SafeEventLog(caller_list)

caller_list.append("added by the caller")
safe.add("added by the log")

print("")
print("COPYING version:")
print("   caller_list:  ", caller_list)
print("   log.entries:  ", safe.entries)
print("   same object?  ", safe.entries is caller_list)

print("")
print("Neither is wrong. But the class must DOCUMENT which it does,")
print("because the caller cannot tell from the call site.")

## Returning internal state creates an alias outward

The mirror image: handing out a reference to your own data lets callers change it.

In [ ]:
class Inventory:
    """Exposes its internal list directly."""

    def __init__(self):
        self._items = ["widget"]

    def get_items(self):
        # Returning the actual list hands out a mutable reference.
        return self._items

    def get_items_safely(self):
        # Returning a copy protects the internal state.
        return list(self._items)


store = Inventory()

# A caller can modify the internal state through the returned list.
leaked = store.get_items()
leaked.append("injected by the caller")

print("After mutating the returned list:")
print("   internal state:", store._items, "<- modified from outside")

# The safe version cannot be tampered with.
store = Inventory()
protected = store.get_items_safely()
protected.append("injected by the caller")

print("")
print("With a defensive copy:")
print("   internal state:", store._items, "<- intact")
print("   what the caller got:", protected)

print("")
print("A tuple makes the intent explicit - the caller CANNOT modify it:")
print("   tuple(store._items) ->", tuple(store._items))

## Detecting unintended aliasing

When data changes and you cannot see why, these three checks find it fast.

In [ ]:
# Set up a situation with hidden sharing.
template = {"role": "user", "permissions": []}
account_a = template
account_b = template

account_a["permissions"].append("read")

print("CHECK 1 - compare identity directly")
print("   account_a is account_b:", account_a is account_b)
print("   account_a is template: ", account_a is template)

print("")
print("CHECK 2 - compare ids")
print("   id(account_a):", id(account_a))
print("   id(account_b):", id(account_b))
print("   id(template): ", id(template))

import sys

print("")
print("CHECK 3 - count references")
print("   references to that dict:", sys.getrefcount(template) - 1)
print("   more than one name points at it, so any of them can change it")

print("")
print("The fix here is to build a fresh dict per account:")

def new_account():
    """Return an independent account dictionary."""
    return {"role": "user", "permissions": []}


independent_a = new_account()
independent_b = new_account()
independent_a["permissions"].append("read")

print("   independent_a:", independent_a)
print("   independent_b:", independent_b)
print("   same object?  ", independent_a is independent_b)

## When aliasing is the point

Aliasing is not something to avoid. Several common patterns depend on it.

In [ ]:
# 1. A shared registry that several parts of a program update.
registry = {}

def register(name, value):
    """Add an entry to the shared registry."""
    # This mutates the module-level dict - deliberately.
    registry[name] = value


register("alpha", 1)
register("beta", 2)
print("1. Shared registry:", registry)

# 2. An accumulator passed into a helper.
def collect_errors(record, errors):
    """Append any problems found to the caller's list."""
    if not record.get("name"):
        errors.append("missing name")
    if not record.get("email"):
        errors.append("missing email")


problems = []
collect_errors({"name": "Asha"}, problems)
collect_errors({}, problems)

print("")
print("2. Accumulator pattern:", problems)

# 3. Two views onto one dataset.
measurements = [10, 20, 30]
latest_view = measurements       # deliberately an alias

measurements.append(40)
print("")
print("3. Live view:", latest_view, "<- stays in sync by design")

print("")
print("In each case the sharing is INTENTIONAL and documented.")
print("That is the difference between a feature and a bug.")

## Takeaways

1. **Aliasing** is two or more names for one object. Assignment, containers,
   arguments, loop variables and dict values all create aliases.
2. Aliasing an **immutable** object is always safe. Aliasing a **mutable** one
   creates another route to change it.
3. A change is visible through every alias **only when you mutate**, never when
   you rebind.
4. Aliased and copied code behave identically until the first mutation — which is
   why these bugs surface late.
5. Storing a constructor argument creates an alias; **copy it** if the object
   should be independent.
6. Returning internal state hands out a mutable reference; return a **copy** or a
   **tuple** to protect it.
7. Diagnose with `is`, `id()` and `sys.getrefcount()`.
8. Intentional aliasing is a legitimate pattern — registries, accumulators, live
   views. Document it.

## Try it yourself

1. Create a list, alias it four different ways, mutate through one, and confirm
   all four see it.
2. Write a class that stores a list argument. Prove the caller can modify its
   internals, then fix it with a defensive copy.
3. In a loop over a list of lists, rebind the loop variable in one version and
   mutate it in another. Explain the difference.
4. Use `sys.getrefcount()` to find how many references a shared dict has.
5. Find a place in your own code where you return internal state. Should it be a
   copy?